# Unit commitment
Dans ce notebook nous adressons le problème de la planification de production d'unités de production électrique constitués de centrales thermiques et d'un bassin hydrographique.

La desciption du problème est donnée dans deux fichiers NetCDF. Le premier décrit les unités de production thermiques et la demande, le second décrit le bassin hydrographique et les turbines. Cette partie est gérée par le module python netCDF4 (https://unidata.github.io/netcdf4-python/).

La formulation du problème se fait à l'aide du module python pulp (https://coin-or.github.io/pulp/). Le problème est formulé sous la forme d'un problème MILP.

La résolution du problème peut se faire soit à l'aide de l'algorithme CBC fourni par défaut avec pulp, soit de l'algorithme GLPK (https://www.gnu.org/software/glpk/) qui est assez ancien (pas d'évolution depuis 2012) et de ce fait moins performant, soit de l'algorithme HiGHS (https://highs.dev/) qui est particulièrement performant.

L'exploitation des résultats se fait à l'aide d'une animation de la production des différentes unités de production via le module matplotlib (https://matplotlib.org/), ainsi qu'un export des résultats dans un fichier tableur via le module pandas (https://pandas.pydata.org/) pour une exploitation par des outils externes.

In [1]:
import numpy as np
import netCDF4 as ncd
import pulp

nom_donnes_thermiques = "200_0_11_w.nc4"
nom_donnes_hydro = "Aurland_1000.nc4"
donnees_thermique = ncd.Dataset(nom_donnes_thermiques).groups["Block_0"]
donnees_hydro = ncd.Dataset(nom_donnes_hydro).groups["Block_0"].groups

Restricted license - for non-production use only - expires 2026-11-23
Restricted license - for non-production use only - expires 2026-11-23
Restricted license - for non-production use only - expires 2026-11-23
Restricted license - for non-production use only - expires 2026-11-23
Restricted license - for non-production use only - expires 2026-11-23
Restricted license - for non-production use only - expires 2026-11-23


In [2]:
# Données thermiques

nb_centrales = len(donnees_thermique.dimensions["NumberUnits"])
T = len(donnees_thermique.dimensions["TimeHorizon"])
demande = list(donnees_thermique["ActivePowerDemand"][:])
cost = []
startupcost = []
pmin = []
pmax = []
MinUpTime = []
MinDownTime = []
production_initiale = []
DeltaRampUp = []
DeltaRampDown = []

for cle in donnees_thermique.groups.keys():
    groupe = donnees_thermique.groups[cle]
    if hasattr(groupe, "type"):
       if ( groupe.type == "ThermalUnitBlock" ):
            cost.append(float(groupe["LinearTerm"][:]))
            startupcost.append(float(groupe["StartUpCost"][:]))
            MinUpTime.append(int(groupe["MinUpTime"][:]))
            MinDownTime.append(int(groupe["MinDownTime"][:]))
            production_initiale.append(float(groupe["InitialPower"][:]))
            DeltaRampDown.append(float(groupe["DeltaRampDown"][:]))
            DeltaRampUp.append(float(groupe["DeltaRampUp"][:]))
            pmin.append(float(groupe["MinPower"][:]))
            pmax.append(float(groupe["MaxPower"][:]))
nb_reservoirs = len(donnees_hydro["UnitBlock_0"].dimensions["NumberReservoirs"])
vmax = list(donnees_hydro["UnitBlock_0"]["MaxVolumetric"][:])
vinit = list(donnees_hydro["UnitBlock_0"]["InitialVolumetric"][:])
a = [_[0] for _ in donnees_hydro["UnitBlock_0"]["Inflows"][:].tolist()]
maxflow = donnees_hydro["UnitBlock_0"]["MaxFlow"][:].tolist()[0]
coefprod = list(donnees_hydro["UnitBlock_0"]["LinearTerm"][:])
nb_turbines = len(donnees_hydro["UnitBlock_0"].dimensions["NumberArcs"])
StartArc = list(donnees_hydro["UnitBlock_0"]["StartArc"][:])
EndArc = list(donnees_hydro["UnitBlock_0"]["EndArc"][:])

## Formulation MILP du problème

On note $N_{th}$ le nombre d'unités thermiques, $N_{res}$ le nombre de réservoirs, $N_{tur}$ le nombre de turbines et $T$ le nombre d'instants à traiter.

On introduit les variables suivantes:
  * $u_{t,i}\in\{0,1\}$ est la variable indiquant si l'unité thermique $i\in\{0,\dots,N_{th}-1\}$ est active ($u_{t,i}=1$) à l'instant $t\in\{0,\dots,T-1\}$;
  * $s_{t,i}\in\{0,1\}$ est la variable indiquant si l'unité thermique $i\in\{0,\dots,N_{th}-1\}$ vient de démarrer ($s_{t,i}=1$) à l'instant $t\in\{0,\dots,T-1\}$;
  * $p_{t,i}\in[0,+\infty[$ est la variable représentant la production de l'unité thermique $i\in\{0,\dots,N_{th}-1\}$ et hydro $i=N_{th}$ à l'instant $t\in\{0,\dots,T-1\}$;
  * $v_{t,j}\in[0,+\infty[$ est la variable représentant le volume du réservoir $j\in\{0,\dots,N_{res}-1\}$ à l'instant $t\in\{0,\dots,T-1\}$;
  * $f_{t,k}\in[0,+\infty[$ est la variable représentant le flux de la turbine $k\in\{0,\dots,N_{tur}-1\}$ à l'instant $t\in\{0,\dots,T-1\}$.

La production $p_{t,i}$ de l'unité thermique $i$ à l'instant $t$ doit vérifier les contraintes suivantes:
  * $p_{0,i}=p_{ini,i}$ qui exprime la valeur initiale de la production de l'unité;
  * $p_{t,i}\geq p_{min, i} u_{t,i}$ qui exprime que la production est supérieure ou égale à $p_{min, i}$ si l'unité est active ou supérieure ou égale à zéro si elle n'est pas active;
  * $p_{t,i}\leq p_{max, i} u_{t,i}$ qui exprime que la production est inférieure ou égale à $p_{max, i}$ si l'unité est active ou supérieure ou égale à zéro si elle n'est pas active;
  * $p_{t,i}-p_{t-1,i}\leq \delta_{up}$ pour $t>0$, qui exprime que l'augmentation de production par rapport à l'instant précédent est majorée par $\delta_{up}$;
  * $p_{t,i}-p_{t-1,i}\geq -\delta_{down}$ pour $t>0$, qui exprime que la diminution de production par rapport à l'instant précédent est majorée par $-\delta_{down}$;

La variable d'allumage $u_{t,i}$ de l'unité thermique $i$ à l'instant $t$ doit vérifier les contraintes suivantes:
  * $u_{0,i}=1$ si $p_{ini,i}>0$, $u_{0,i}=0$ sinon, qui exprime que les unités thermiques peuvent déjà être en fonctionnement à l'instant initial;
  * $\forall k\in\{0,T-T_{up}-1\}, \sum_{t=k}^{k+T_{up}-1}u_{t,i}\geq T_{up}(u_{k+1,i}-u_{k,i})$, qui exprime que si l'unité est active à l'instant $k$ elle doit le rester pendant un minimum de $T_{up}$ instants;
  * $\forall k\in\{0,T-T_{down}-1\}, \sum_{t=k}^{k+T_{down}-1}u_{t,i}\leq T_{down}(1-(u_{k+1,i}-u_{k,i}))$, qui exprime que si l'unité est inactive à l'instant $k$ elle doit le rester pendant un minimum de $T_{down}$ instants.

La variable de démarrage $s_{t,i}$ de l'unité thermique $i$ à l'instant $t$ doit vérifier la contrainte $s_{t,i}\geq u_{t,i}-u_{(t-1)^+,i}$ qui exprime que l'unité vient de s'allumer si l'unité est passée d'inactive à active.

Le volume $v_{t,j}$ du réservoir $j$ à l'instant $t$ doit vérifier les contraintes suivantes:
  * $v_{t,j}\in[0,v_{max,j}]$ qui exprime que le volume est positif et majoré par $v_{max,j}$;
  * $v_{0,j}=v_{ini, j}$ qui exprime la valeur initiale du volume du réservoir;
  * $v_{t,j}=v_{t-1,j}+a_j+\sum_{m\in E_m}f_{t,m}-\sum_{n\in E_n}f_{t,n}$ pour $t>0$ qui exprime l'évolution du volume du réservoir du fait de l'apport $a_j$ du fait de l'hydrologie, du volume turbiné depuis les réservoirs $m\in E_m$ débouchant dans le réservoir $j$ auquel on retranche le volume turbiné vers les réservoirs $n\in E_n$ dans lesquels le réservoir $j$ peut se déverser.

La production $p_{t,N_{th}}$ de l'unité hydro vérifie $p_{t,N_{th}}=\sum_{k=0}^{N_{tur}-1}c_kf_{t,k}$ où $c_k$ est le coefficient de production de la turbine $k\in\{0,\dots,N_{tur}-1\}$.

On a enfin la contrainte liée à la demande: $\sum_{i=0}^{N_{th}}p_{t,i}\geq d_{t}$ où $d_t$ est la demande à l'instant $t\in\{0,\dots,T-1\}$.

L'objectif est de trouver un ensemble de valeurs pour ces variables compatible avec l'ensemble des contraintes et minimisant la fonction de coût suivante (la production hydro est gratuite):
$$
J(u_{t,i}, s_{t,i}, p_{t,i}, v_{t,j}, f_{t,k})=\sum_{t=0}^{T-1}\sum_{i=0}^{N_{th}-1}\left(C_{prod,i}p_{t,i}+C_{start,i}s_{t,i}\right)
$$
où $C_{prod,i}$ est le coût associé à une production $p_{t,i}$ de l'unité thermique $i$ et $C_{start,i}$ le coût associé au démarrage de cette unité thermique.


In [3]:
# Variables
# u indique pour chaque centrale thermique si elle est active
u = [[pulp.LpVariable("u_t" + str(t) + "_i" + str(i), cat="Binary") for i in range(nb_centrales)] for t in range(T)]
# s indique pour chaque centrale thermique si elle vient de s'allumer, pour compter le surcoût d'allumage
s = [[pulp.LpVariable("s_t" + str(t) + "_i" + str(i), cat="Binary") for i in range(nb_centrales)] for t in range(T)]
# Production des centrales thermiques et de l'unité hydro (en fin de liste)
p = [[pulp.LpVariable("p_t" + str(t) + "_i" + str(i)) for i in range(nb_centrales + 1)] for t in range(T)]
# v est le volume occupé de chaque réservoir à chaque pas de temps
v = [[pulp.LpVariable("v_t" + str(t) + "_i" + str(i)) for i in range(nb_reservoirs)] for t in range(T)]
# f est le flux de chaque turbine à chaque pas de temps
f = [[pulp.LpVariable("f_t" + str(t) + "_i" + str(i)) for i in range(nb_turbines)] for t in range(T)]
# Coût total
J = pulp.LpVariable("J")

# Définir le modèle
modele = pulp.LpProblem("UnitCommitment", pulp.LpMinimize)

### Contraintes ###
## Contraintes thermiques ##
# Contraintes sur la production initiale des unités thermiques
for i in range(nb_centrales):
    p_ini_i = production_initiale[i]
    # On ne considère que les conditions initiales indiquant que
    # l'unité est active sinon on va marquer les autres unités
    # comme étant éteintes et devant le rester durant MinDownTime
    # ce qui rend le problème infaisable dans certaines instances
    # Utiliser
    # if p_ini_i > 0.0 and False:
    # pour tester l'effet de l'initialisation sur la solution
    if p_ini_i > 0.0:
        modele += p[0][i] == p_ini_i
        modele += u[0][i] == 1

for t in range(T):
    modele += sum(p[t][:]) >= demande[t]
    for i in range(nb_centrales):
        # Contraintes sur les bornes de production
        modele += p[t][i] - pmin[i] * u[t][i] >= 0.0
        modele += p[t][i] - pmax[i] * u[t][i] <= 0.0

        # Contraintes sur le surcoût d'allumage
        modele += s[t][i] >= u[t][i] - u[max(0, t-1)][i]

        # Contrainte de gradient
        if t > 0:
            modele +=  DeltaRampUp[i]   >= p[t][i] - p[t-1][i]
            modele += -DeltaRampDown[i] <= p[t][i] - p[t-1][i]

# Contraintes sur MinUpTime et MinDownTime
for i in range(nb_centrales):
    for k in range(T - MinUpTime[i]):
        modele += sum(u[t][i] for t in range(k, k + MinUpTime[i])) >= MinUpTime[i] * (u[k][i] - u[k-1][i])

for i in range(nb_centrales):
    for k in range(T - MinDownTime[i]):
        modele += sum(u[t][i] for t in range(k, k + MinDownTime[i])) <= MinDownTime[i] * (1 - (u[k][i] - u[k-1][i]))


## Contraintes hydrauliques ##
# Equilibre énergetique
# Contraintes sur les bornes de volume et de flux
for t in range(T):
    for i in range(nb_reservoirs):
        modele += v[t][i] <= vmax[i]
        modele += v[t][i] >= 0.0
    for i in range(nb_turbines):
        modele += f[t][i] <= maxflow[i]
        modele += f[t][i] >= 0.0

# Contraintes sur le volume initial des reservoirs
for i in range(nb_reservoirs):
    modele += v[0][i] == vinit[i]

# Actualisation des volumes
for t in range(1, T):
    for i in range(nb_reservoirs):
        # Indice des turbines débouchant dans le réservoir i
        lm = [km for km, xm in enumerate(EndArc) if xm == i]
        # Indice des turbines pompant dans le réservoir i
        ln = [kn for kn, xn in enumerate(StartArc) if xn == i]
        modele += v[t][i] == v[t-1][i] + a[i] + sum(f[t][m] for m in lm) - sum(f[t][n] for n in ln)

# Calcul de la production hydro
for t in range(T):
    modele += p[t][nb_centrales] == sum([coefprod[k]*f[t][k] for k in range(nb_turbines)])

# Fonction objectif. On ne somme que sur les centrales thermiques puisque la production hydro est gratuite
modele += J == sum([cost[i] * p[t][i] + startupcost[i] * s[t][i] for i in range(nb_centrales) for t in range(T)])
modele += J

In [4]:
# Exporter le modèle au format LP
variables = modele.writeLP("modele.lp")

In [5]:
# Résoudre le problème
solver = pulp.HiGHS_CMD(keepFiles=False, mip=True, msg=True, options=None, timeLimit=None, gapRel=None, gapAbs=None, threads=32, logPath=None, warmStart=False)
#solver = pulp.GLPK_CMD(keepFiles=False, mip=True, msg=True, options=None, timeLimit=None)
status = modele.solve(solver)

Running HiGHS 1.12.0 (git hash: n/a): Copyright (c) 2025 HiGHS under MIT licence terms
Set option parallel to "on"
Set option threads to 32
Set option log_file to "/tmp/005ce1164141413683df50d3dec7dca0-pulp.HiGHS_log"
Set option write_solution_to_file to true
Set option solution_file to "/tmp/005ce1164141413683df50d3dec7dca0-pulp.sol"
Number of BV entries in BOUNDS section is 9600
MIP 005ce1164141413683df50d3dec7dca0-pulp has 32446 rows; 14857 cols; 104120 nonzeros; 9600 integer variables (9600 binary)
Coefficient ranges:
  Matrix  [1e+00, 2e+03]
  Cost    [1e+00, 1e+00]
  Bound   [1e+00, 1e+00]
  RHS     [6e-04, 3e+04]
Presolving model
28064 rows, 14157 cols, 87242 nonzeros  0s
22926 rows, 11212 cols, 50830 nonzeros  0s
15220 rows, 9141 cols, 42223 nonzeros  0s
13822 rows, 8062 cols, 37618 nonzeros  0s
Presolve reductions: rows 13822(-18624); columns 8062(-6795); nonzeros 37618(-66502) 

Solving MIP model with:
   13822 rows
   8062 cols (3292 binary, 0 integer, 0 implied int., 4770 c

In [6]:
# Export de la solution
import pandas as pd
prefixe = nom_donnes_thermiques.split(".")[0] + "_" + nom_donnes_hydro.split(".")[0] + "_"
res_u = pd.DataFrame([[pulp.value(u[t][i]) for i in range(nb_centrales)] for t in range(T)], columns=["centrale_u_" + str(i) for i in range(nb_centrales)])
res_u.to_excel(prefixe + "u.xlsx")
res_s = pd.DataFrame([[pulp.value(s[t][i]) for i in range(nb_centrales)] for t in range(T)], columns=["centrale_s_" + str(i) for i in range(nb_centrales)])
res_s.to_excel(prefixe + "s.xlsx")
res_p = pd.DataFrame([[pulp.value(p[t][i]) for i in range(nb_centrales)] for t in range(T)], columns=["centrale_p_" + str(i) for i in range(nb_centrales)])
res_p.to_excel(prefixe + "p.xlsx")
res_v = pd.DataFrame([[pulp.value(v[t][i]) for i in range(nb_reservoirs)] for t in range(T)], columns=["reservoir_v_" + str(i) for i in range(nb_reservoirs)])
res_v.to_excel(prefixe + "v.xlsx")
res_f = pd.DataFrame([[pulp.value(f[t][i]) for i in range(nb_turbines)] for t in range(T)], columns=["turbine_v_" + str(i) for i in range(nb_turbines)])
res_f.to_excel(prefixe + "f.xlsx")
res_p_hydro = pd.DataFrame([[pulp.value(p[t][nb_centrales])] for t in range(T)], columns=["production_hydro_"])
res_p_hydro.to_excel(prefixe + "p_hydro.xlsx")

In [7]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation
from IPython.display import HTML

# Valeur de la programmation
Jopt = pulp.value(J) / 1000
# Couleurs pour les barres
cmap_th = plt.get_cmap('inferno')
cmap_hy = plt.get_cmap('viridis')

# Grille 2x2:
#            |
# Production | Demande vs
# par unité  | production
# -----------+-----------
#  Volumes   | Flux
#  réservoirs| turbines
#            |
fig, ax = plt.subplots(nrows=2, ncols=2, gridspec_kw={'hspace': 0.3, 'wspace': 0.3})
fig.set_figwidth(12)
fig.set_figheight(6)
# Bornes sur la production
pMin = min(0.0, 100.0*(int(min([pulp.value(p[t][i]) for t in range(T) for i in range(nb_centrales+1)])/100)))
pMax = 100.0*(int(max([pulp.value(p[t][i]) for t in range(T) for i in range(nb_centrales+1)])/100)+1)
ax[0,0].axis([0.5, nb_centrales + 1.5, pMin, pMax])
ax[0,0].set_xlabel("Numéro d'unité")
ax[0,0].set_ylabel(r"Puissance ($MW$)")
# Bornes sur les volumes
vMin = min(0.0, 100.0*(int(min([pulp.value(v[t][i]) for t in range(T) for i in range(nb_reservoirs)])/100)))
vMax = 100.0*(int(max([pulp.value(v[t][i]) for t in range(T) for i in range(nb_reservoirs)])/100)+1)
ax[1,0].axis([0.5, nb_reservoirs+0.5, vMin, vMax])
ax[1,0].set_xlabel("Numéro de réservoir")
ax[1,0].set_ylabel(r"Volume ($m^3$)")
# Bornes sur les flux
fMin = min(0.0, 1.0*(int(min([pulp.value(f[t][i]) for t in range(T) for i in range(nb_turbines)])/1)))
fMax = 0.5*(int(max([pulp.value(f[t][i]) for t in range(T) for i in range(nb_turbines)])/0.5)+1)
ax[1,1].axis([0.5, nb_turbines+0.5, fMin, fMax])
ax[1,1].set_xlabel("Numéro de turbine")
ax[1,1].set_ylabel(r"Débit ($m^3.h^{-1}$)")
# Courbe de production totale
productionTotale = [sum([pulp.value(p[t][i]) for i in range(nb_centrales+1)]) for t in range(T)]
ax[0,1].axis([1, T, 0.0, 200.0*(int(max(productionTotale)/200)+5)])
ax[0,1].set_xlabel(r"Temps ($h$)")
ax[0,1].set_ylabel(r"Puissance ($MW$)")

#############################################
# Barres pour les productions individuelles #
#############################################
barcollectionProduction = ax[0,0].bar(range(1, nb_centrales+2), [0.0 for _ in range(nb_centrales+1)],
                                      color=[cmap_th(0.5 + 0.5*i/nb_centrales) for i in range(nb_centrales)] + ["aqua"])
# Texte uniquement si peu d'unités de production
if nb_centrales <= 20:
    textProductions = [ax[0,0].text(b.get_x() + b.get_width() / 2.0, 0.05 * pMax, "",
                                   color='black', ha='center', va='bottom', rotation='vertical') for i,b in enumerate(barcollectionProduction)]
titleTh = ax[0,0].text(0.5,1.1, "", bbox={'facecolor':'w', 'alpha':0.5, 'pad':5},
                transform=ax[0,0].transAxes, ha="center")

##############################
# Barres pour les réservoirs #
##############################
barcollectionReservoirs = ax[1,0].bar(range(1, nb_reservoirs+1), [0.0 for _ in range(nb_reservoirs)],
                                      color=[cmap_hy(0.5*i/nb_reservoirs) for i in range(nb_reservoirs)])
# Texte uniquement si peu de réservoirs
if nb_reservoirs <= 20:
    textReservoirs = [ax[1,0].text(b.get_x() + b.get_width() / 2.0, 0.05 * vMax, "", color='red', ha='center', va='bottom',
                                   rotation='vertical') for b in barcollectionReservoirs]
titleReservoirs = ax[1,0].text(0.5,0.8, "", bbox={'facecolor':'w', 'alpha':0.5, 'pad':5},
                transform=ax[1,0].transAxes, ha="center")

############################
# Barres pour les turbines #
############################
barcollectionTurbines = ax[1,1].bar(range(1, nb_turbines+1), [0.0 for _ in range(nb_turbines)],
                                    color=[cmap_hy(0.5+0.5*i/nb_turbines) for i in range(nb_turbines)])
if nb_turbines <= 20:
    textTurbines = [ax[1,1].text(b.get_x() + b.get_width() / 2.0, 0.05 * fMax, "", color='red', ha='center', va='bottom',
                                 rotation='vertical') for b in barcollectionTurbines]
titleTurbines = ax[1,1].text(0.5,0.8, "", bbox={'facecolor':'w', 'alpha':0.5, 'pad':5},
                transform=ax[1,1].transAxes, ha="center")

##############################
# Courbes production/demande #
##############################
curve = ax[0,1].plot(list(range(1, T+1)), demande, label="demande", color="red")
curve = ax[0,1].plot(list(range(1, T+1)), productionTotale, label="production", color="black")
ax[0,1].legend()
marker_demande, = ax[0,1].plot(0,1, marker="o", markerfacecolor='none', markeredgecolor='r')
marker_production, = ax[0,1].plot(0,1, marker="x", color="black")
titleProd = ax[0,1].text(0.5,1.1, "Profils demande/production, coût=%.0fk€" % Jopt, bbox={'facecolor':'w', 'alpha':0.5, 'pad':5},
                transform=ax[0,1].transAxes, ha="center")

# Fonction d'animation
def animate(n):
    # On commence à 1h
    t = n % T + 1
    marker_demande.set_data([t], [demande[t-1]])
    marker_production.set_data([t], [productionTotale[t-1]])
    titleTh.set_text("Centrales thermiques et hydro\nProduction t = {}h".format(t))
    for i, b in enumerate(barcollectionProduction):
        value = pulp.value(p[n % T][i])
        b.set_height(value)
        if nb_centrales <= 20:
            nom = "Th_" + str(i) if i < nb_centrales else "Hydro"
            if value > 0.0:
                nom += " = {:.4g}".format(value)
                textProductions[i].set_text(nom)
    titleReservoirs.set_text("Volume réservoir\nt = {}h".format(t))
    for i, b in enumerate(barcollectionReservoirs):
        value = pulp.value(v[n % T][i])
        b.set_height(value)
        if nb_reservoirs <= 20:
            textReservoirs[i].set_text("{:.4g}".format(value))
    titleTurbines.set_text("Débit turbine\nt = {}h".format(t))
    for i, b in enumerate(barcollectionTurbines):
        value = pulp.value(f[n % T][i])
        b.set_height(value)
        if nb_turbines <= 20:
            textTurbines[i].set_text("{:.4g}".format(value))

# Animation au format jshtml pour être insérée dans le notebook
ani = matplotlib.animation.FuncAnimation(fig, animate, frames=T)
plt.close()

HTML(ani.to_jshtml(default_mode='once'))